# Phase 7 — Strategy selection

Module 2 asked **whether** to augment. This asks **which** structural signal to add.

Observe the winners → derive one candidate rule → test the whole two-stage framework on unseen graphs.

GraphSAGE, K=10, link prediction, 10 seeds (42–51). Variants are ranked by their **mean** over the seeds.

In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from experiments import strategy_select as ss
from virgo import frozen_rules as fr

# Same table style as notebook 6: fixed layout + wrapped headers, so a wide table never scrolls out of the output area.
STYLE = [
    {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
    {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
    {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
]
print(f"panel ({len(ss.PANEL)}): {ss.PANEL}")
print(f"held out ({len(fr.STRATEGY_HELDOUT)}): {fr.STRATEGY_HELDOUT}")
print("seeds 42-51 | ranking = MEAN over seeds (std/sem kept in the CSVs, not used to rank)")

panel (14): ['cora', 'enzymes', 'roman_empire', 'tolokers', 'questions', 'squirrel_filtered', 'amazon_ratings', 'amazon_photo', 'lastfm_asia', 'pubmed', 'actor', 'minesweeper', 'citeseer_linqs', 'proteins']
held out (3): ['amherst41', 'johnshopkins55', 'cornell5']
seeds 42-51 | ranking = MEAN over seeds (std/sem kept in the CSVs, not used to rank)


/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Which signal wins on the discovery panel

The original graph against the best augmented variant, per dataset. This is what the rule is derived from.

In [2]:
display(ss.panel_table(ss.PANEL).style.set_table_styles(STYLE).hide(axis="index"))

Dataset,Original score,Best augmented variant,Best score,Winning signal
actor,0.597000,degree,0.664900,degree
amazon_photo,0.791500,hybrid_centrality,0.782900,original
amazon_ratings,0.757100,hybrid_centrality,0.692900,original
citeseer_linqs,0.627900,hybrid_degree,0.566400,original
cora,0.618900,hybrid_centrality,0.585400,original
enzymes,0.698600,hybrid_centrality,0.666600,original
lastfm_asia,0.715900,hybrid_centrality,0.728400,centrality
minesweeper,0.713300,hybrid,0.668400,original
proteins,0.673900,hybrid_centrality,0.589500,original
pubmed,0.628300,hybrid_centrality,0.653300,centrality


## 2 · The candidate rule

One rule survived the screen over seven graph properties × four signals. The rejected candidates are in `results/strategy_patterns.csv`; the full screen is in the appendix.

**Status: fitted, not validated** — every labelled dataset is in the panel, so the test is §3. The fit used a sem tie band (part of the frozen rule); §1 and §3 rank by mean.

In [3]:
display(ss.rule_table().style.set_table_styles(STYLE).hide(axis="index"))

Rule,Condition,Prediction,Fitted datasets,Evidence
strategy1,nbr_predictability_adjusted > 0.0092 (interval 0.006-0.0123),"centrality, else psi or degree (undetermined) - only when augmentation is already indicated","7 augmenting, of 14 in the panel","0 exceptions, rho 0.866, LOO 5/7 vs 0.5714 majority; sem band only"


## 3 · Held-out test — the two-stage framework

**Stage 1** = the frozen augment call (`predict_gated`). **Stage 2** = the rule above, consulted only where stage 1 says *augment*.

Both predictions were written to disk **before** any encoder ran.

In [4]:
from experiments import predict_strategy as ps

HELDOUT = ["chameleon_filtered", "texas", "twitch_pt",                      # withdrawn 2026-08-14, scores kept
           "reed98", "amherst41", "johnshopkins55", "cornell5"]            # LINKX / Facebook100
route = ps.routing(HELDOUT)
display(route.round(4).style.set_table_styles(STYLE).hide(axis="index"))

dataset,homophily_adjusted,original_retention,stage1_call,nbr_predictability_adjusted,stage2_ran,stage2_prediction
chameleon_filtered,nan,nan,not run (batch predates stage 1),0.125000,True,centrality
texas,nan,nan,not run (batch predates stage 1),0.280500,True,centrality
twitch_pt,nan,nan,not run (batch predates stage 1),0.142200,True,centrality
reed98,0.021900,0.023700,keep original,nan,False,skipped - stage 1 says keep original
amherst41,0.059800,0.009200,augment,0.212800,True,centrality
johnshopkins55,0.097200,0.005500,augment,0.227800,True,centrality
cornell5,0.090700,0.002000,augment,0.276000,True,centrality


### 3.1 · Stage 2 — the winner against the runner-up

Highest mean wins, however small the gap.

Stage 1 was only *measured* on the `Full 2-stage` rows; the earlier batch ran before the protocol existed, so its `KEEP` is inferred from the outcome. `twitch_pt` is in the routing table above — augmentation helped there, but `hybrid_degree` won, not centrality. Margins and seed spreads: `results/module7_outcome.csv`.

In [5]:
out = ps.outcome(list(route.dataset[route.stage2_ran]))
out.to_csv("results/module7_outcome.csv", index=False)                # full detail, including margins and both verdicts
table = ps.summary(route, out)
display(table[table.Dataset != "twitch_pt"].style.set_table_styles(STYLE).hide(axis="index"))

o = out[[c == "augment" for c in route.set_index("dataset").stage1_call[out.dataset]]]
print(f"Pre-registered test (stage 1 said AUGMENT): {int(o.correct_by_margin.sum())}/{len(o)} correct "
      f"({', '.join(o.dataset)})")

Dataset,Test batch,Stage 1: KEEP or AUGMENT?,Stage 2 used?,Stage 2 prediction,Best result,2nd-best result,Final interpretation
chameleon_filtered,Earlier test,KEEP,No,N/A,— (not applicable),— (not applicable),Stage 1 said KEEP → augmentation unnecessary
texas,Earlier test,KEEP,No,N/A,— (not applicable),— (not applicable),Stage 1 said KEEP → augmentation unnecessary
reed98,Full 2-stage,KEEP,No,N/A,— (not applicable),— (not applicable),Stage 1 said KEEP → augmentation unnecessary
amherst41,Full 2-stage,AUGMENT,Yes,Centrality,Centrality 0.6749,Hybrid-centrality 0.6718,Correct: centrality best
johnshopkins55,Full 2-stage,AUGMENT,Yes,Centrality,Centrality 0.7160,Hybrid-centrality 0.7021,Correct: centrality best
cornell5,Full 2-stage,AUGMENT,Yes,Centrality,Centrality 0.7078,Hybrid 0.7050,Correct: centrality best


Pre-registered test (stage 1 said AUGMENT): 3/3 correct (amherst41, johnshopkins55, cornell5)


## 4 · Reading

- **Stage 1 held: 2/2** where the experiment decided, and it is what stopped `reed98`. Rule 1 alone would have said *augment* on all four LINKX graphs.
- **Stage 2 held: 3/3** — centrality has the highest mean on every dataset stage 1 sent to *augment*, and augmentation beat the original graph on all three.
- **Rank, not significance.** Two of those three margins (0.0031, 0.0028) are small enough that a different seed set could reorder the top two; the claim is that centrality *ranks first*, not that it is measurably better than the runner-up.
- **Still untested on its other side.** The cut is 0.0092 and every held-out graph landed far above it, so all six calls were *centrality*. A real falsification test needs an augmenting graph below the cut.

## Appendix · the full rule search

Kept for reproducibility: the winner sets, the seven properties, every signal × property test, and the undetermined ψ-vs-degree zone.

In [6]:
win = ss.winners(ss.PANEL)
display(win.style.set_table_styles(STYLE).hide(axis="index"))

dataset,task,metric,seeds,band,best_variant,best_score,original,winner_set,winner_signals,n_tied,beats_original,signal_decided
actor,link prediction (AUC),auc,10,sem,degree,0.664900,0.597000,degree,degree,1,True,True
amazon_photo,link prediction (AUC),auc,10,sem,original,0.791500,0.791500,original,original,1,False,False
amazon_ratings,link prediction (AUC),auc,10,sem,original,0.757100,0.757100,original,original,1,False,False
citeseer_linqs,link prediction (AUC),auc,10,sem,original,0.627900,0.627900,original,original,1,False,False
cora,link prediction (AUC),auc,10,sem,original,0.618900,0.618900,original,original,1,False,False
enzymes,link prediction (AUC),auc,10,sem,original,0.698600,0.698600,original,original,1,False,False
lastfm_asia,link prediction (AUC),auc,10,sem,hybrid_centrality,0.728400,0.715900,hybrid_centrality,centrality,1,True,True
minesweeper,link prediction (AUC),auc,10,sem,original,0.713300,0.713300,original,original,1,False,False
proteins,link prediction (AUC),auc,10,sem,original,0.673900,0.673900,original,original,1,False,False
pubmed,link prediction (AUC),auc,10,sem,hybrid_centrality,0.653300,0.628300,hybrid_centrality,centrality,1,True,True


In [7]:
prop = ss.properties(ss.PANEL)
display(win[["dataset", "winner_signals", "signal_decided"]].merge(prop, on="dataset")
        .round(4).style.set_table_styles(STYLE).hide(axis="index"))

dataset,winner_signals,signal_decided,homophily_adjusted,nbr_predictability_adjusted,components,largest_component_frac,avg_degree,avg_clustering,n_classes,edge_homophily,nbr_label_entropy,density,degree_gini,degree_skew,degree_assortativity,majority_class_frac,nodes
actor,degree,True,0.002800,0.006000,1,1.000000,7.015500,0.080200,5.000000,0.216700,0.509200,0.000900,0.569100,51.785800,-0.046900,0.258600,7600
amazon_photo,original,False,0.785000,0.903300,136,0.978700,31.132300,0.404000,8.000000,0.827200,0.156300,0.004100,0.518800,10.416700,-0.044900,0.253700,7650
amazon_ratings,original,False,0.140200,0.108700,1,1.000000,7.598400,0.581600,5.000000,0.380400,0.526300,0.000300,0.266900,6.916700,-0.092000,0.367900,24492
citeseer_linqs,original,False,0.673100,0.684900,390,0.646400,2.779400,0.144700,6.000000,0.737700,0.099700,0.000900,0.435300,10.121400,0.048100,0.208600,3264
cora,original,False,0.771100,0.814300,78,0.917700,3.898100,0.240700,7.000000,0.810000,0.110800,0.001400,0.405100,15.271400,-0.065900,0.302100,2708
enzymes,original,False,0.361300,0.484400,640,0.006400,3.828900,0.402400,3.000000,0.665300,0.372100,0.000200,0.156900,0.397600,0.175500,0.494900,19474
lastfm_asia,centrality,True,0.856200,0.849800,1,1.000000,7.294300,0.219400,18.000000,0.873900,0.068100,0.001000,0.583500,5.702400,0.017100,0.206200,7624
minesweeper,original,False,0.009400,0.000000,1,1.000000,7.880400,0.435500,2.000000,0.682800,0.612100,0.000800,0.014600,-4.771600,0.391500,0.800000,10000
proteins,original,False,0.355200,0.476000,1195,0.014300,3.729100,0.381400,3.000000,0.656800,0.363500,0.000100,0.163400,1.009900,0.151600,0.486500,43466
pubmed,centrality,True,0.686000,0.694600,1,1.000000,4.496000,0.060200,3.000000,0.802400,0.126400,0.000200,0.603700,5.209300,-0.043600,0.399400,19717


In [8]:
import pandas as pd
# Two scopes: 'all' = every panel graph; 'augmented' = only where augmentation beats the original,
# which is the conditional question - given that augmenting helps, WHICH signal.
pat = pd.concat([ss.patterns(win, prop, scope=s) for s in ("all", "augmented")], ignore_index=True)
cols = ["scope", "signal", "predictor", "n_scored", "n_wins", "n_sole_wins", "win_range", "rest_range",
        "spearman_rho", "threshold", "win_side", "n_exceptions", "loo_accuracy", "loo_folds",
        "loo_majority", "majority_baseline", "tie_dominated", "credible"]
display(pat[cols].style.set_table_styles(STYLE).hide(axis="index")
        .apply(lambda r: ["background-color: #d6f5d6" if r.credible else ""] * len(r), axis=1))

scope,signal,predictor,n_scored,n_wins,n_sole_wins,win_range,rest_range,spearman_rho,threshold,win_side,n_exceptions,loo_accuracy,loo_folds,loo_majority,majority_baseline,tie_dominated,credible
all,centrality,avg_clustering,14,4,3,0.0602-0.4631,0.0307-0.5816,-0.117700,0.070200,low,4,0.571400,14,0.714300,0.714300,False,False
all,centrality,avg_degree,14,4,3,2.9059-42.2834,2.7794-88.2803,0.000000,3.317500,low,4,0.571400,14,0.714300,0.714300,False,False
all,centrality,components,14,4,3,1.0000-1.0000,1.0000-1195.0000,-0.457100,39.500000,low,5,0.357100,14,0.714300,0.714300,False,False
all,centrality,homophily_adjusted,14,4,3,-0.0468-0.8562,0.0028-0.7850,-0.039200,-0.022000,low,3,0.642900,14,0.714300,0.714300,False,False
all,centrality,largest_component_frac,14,4,3,1.0000-1.0000,0.0064-1.0000,0.457100,0.010300,low,5,0.214300,14,0.714300,0.714300,False,False
all,centrality,n_classes,14,4,3,3.0000-18.0000,2.0000-8.0000,0.398000,13.000000,high,2,0.857100,14,0.714300,0.714300,False,False
all,centrality,nbr_predictability_adjusted,14,4,3,0.0123-0.8498,-0.3274-0.9033,0.235300,0.689700,high,4,0.500000,14,0.714300,0.714300,False,False
all,degree,avg_clustering,14,3,2,0.0307-0.4631,0.0602-0.5816,-0.280700,0.045400,low,2,0.785700,14,0.785700,0.785700,False,False
all,degree,avg_degree,14,3,2,6.2771-42.2834,2.7794-88.2803,0.237500,36.707900,high,3,0.642900,14,0.785700,0.785700,False,False
all,degree,components,14,3,2,1.0000-1.0000,1.0000-1195.0000,-0.377500,917.500000,high,4,0.714300,14,0.785700,0.785700,False,False


In [9]:
ss.report(win, pat)


STEP 4 - WINNING STRATEGY PER DATASET (band = sem; metric differs on ogbl_ddi by protocol)
          dataset metric  seeds  original      best_variant  best_score                   winner_set    winner_signals  n_tied  beats_original  signal_decided
            actor    auc     10    0.5970            degree      0.6649                       degree            degree       1            True            True
     amazon_photo    auc     10    0.7915          original      0.7915                     original          original       1           False           False
   amazon_ratings    auc     10    0.7571          original      0.7571                     original          original       1           False           False
   citeseer_linqs    auc     10    0.6279          original      0.6279                     original          original       1           False           False
             cora    auc     10    0.6189          original      0.6189                     original          ori

In [10]:
print(fr.FROZEN_STRATEGY)
d = win.merge(prop, on="dataset")
d = d[d.beats_original][["dataset", "winner_signals", "nbr_predictability_adjusted", "homophily_adjusted"]]
d["rule_says"] = [fr.predict_strategy({"nbr_predictability_adjusted": v}) for v in d.nbr_predictability_adjusted]
display(d.sort_values("nbr_predictability_adjusted").round(4).style.set_table_styles(STYLE).hide(axis="index"))

Strategy(name='strategy1', signal='centrality', predictor='nbr_predictability_adjusted', op='>', point=0.0092, interval=(0.006, 0.0123), needs_labels=True, applies_when='augmentation is already indicated', fitted_on='STRATEGY_PANEL')


dataset,winner_signals,nbr_predictability_adjusted,homophily_adjusted,rule_says
tolokers,psi,-0.327400,0.092600,psi or degree (undetermined)
questions,degree,-0.004100,0.020700,psi or degree (undetermined)
actor,degree,0.006000,0.002800,psi or degree (undetermined)
squirrel_filtered,centrality|degree,0.012300,0.008600,centrality
roman_empire,centrality,0.332400,-0.046800,centrality
pubmed,centrality,0.694600,0.686000,centrality
lastfm_asia,centrality,0.849800,0.856200,centrality
